# LeetCode 3774 — Absolute Difference Between Maximum and Minimum K Elements bằng Python

Link đề: https://leetcode.com/problems/absolute-difference-between-maximum-and-minimum-k-elements/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Heap

Skeleton LeetCode:

```python
class Solution:
    def absDifference(self, nums: List[int], k: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org (một nguồn mirror ghi nhầm tên khác cho ID này; trang doocs cho đúng đề).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính **tổng của `k` phần tử lớn nhất** trừ **tổng của `k` phần tử nhỏ nhất**, rồi lấy giá trị tuyệt đối.
2. Biết "k phần tử lớn nhất / nhỏ nhất" trong Python: sắp xếp + slicing, `heapq.nlargest/nsmallest`, và đếm bằng mảng đếm vì giá trị chỉ từ `1` đến `100`.
3. Hiểu **giá trị trùng được tính riêng** (mỗi vị trí là một phần tử), và vì sao không được dùng `set` ở đây.
4. Viết 5 cách khác nhau (loại dần max/min, sắp xếp, `heapq`, mảng đếm, hai con trỏ trên mảng đã sắp xếp) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: khử trùng bằng `set`, quên sắp xếp trước khi cắt, so sánh phần tử thứ `k` thay vì **tổng** `k` phần tử, và bỏ quên `abs`/thứ tự trừ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Tính **hiệu tuyệt đối** giữa:

- tổng của `k` phần tử **lớn nhất** trong `nums`, và
- tổng của `k` phần tử **nhỏ nhất** trong `nums`.

Trả về hiệu này (một số nguyên không âm). Mỗi vị trí trong mảng là một phần tử riêng, kể cả khi giá trị trùng nhau.

| Ví dụ | nums, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[5,2,2,4]`, `2` | `5` | Hai lớn nhất `5, 4` (tổng 9), hai nhỏ nhất `2, 2` (tổng 4), `|9 - 4| = 5` |
| 2 | `[100]`, `1` | `0` | Lớn nhất và nhỏ nhất đều là `100`, hiệu `0` |

Constraints: `1 <= n == nums.length <= 100`, `1 <= nums[i] <= 100`, `1 <= k <= n`.

In [1]:
for nums, k in [([5, 2, 2, 4], 2), ([100], 1)]:
    s = sorted(nums)
    big, small = s[-k:], s[:k]
    print(f"nums={nums} k={k}  sắp xếp={s}  k lớn nhất={big} (tổng {sum(big)})  k nhỏ nhất={small} (tổng {sum(small)})  -> {abs(sum(big) - sum(small))}")

nums=[5, 2, 2, 4] k=2  sắp xếp=[2, 2, 4, 5]  k lớn nhất=[4, 5] (tổng 9)  k nhỏ nhất=[2, 2] (tổng 4)  -> 5
nums=[100] k=1  sắp xếp=[100]  k lớn nhất=[100] (tổng 100)  k nhỏ nhất=[100] (tổng 100)  -> 0


## 3. Kiến thức nền

### 3.1. `sorted` và slicing đầu/cuối

`sorted(nums)` trả list mới tăng dần (không sửa `nums`). Với `s` đã sắp xếp: `s[:k]` là `k` phần tử **nhỏ nhất**, `s[-k:]` là `k` phần tử **lớn nhất**. Chú ý `s[-k:]` với `k >= 1` luôn lấy đúng `k` phần tử cuối (nhưng `k = 0` sẽ lấy cả list, bài này `k >= 1`).

In [2]:
s = sorted([5, 2, 2, 4])
print(s, s[:2], s[-2:])
print(s[:1], s[-1:])
print(s[-0:], "<- k=0 lấy cả list (bẫy), đề cho k >= 1")

[2, 2, 4, 5] [2, 2] [4, 5]
[2] [5]
[2, 2, 4, 5] <- k=0 lấy cả list (bẫy), đề cho k >= 1


### 3.2. Giá trị tuyệt đối `abs`

`abs(x)` trả về giá trị không âm của `x`. Khi lấy `tổng_lớn - tổng_nhỏ` thì kết quả luôn `>= 0` vì `k` phần tử lớn nhất có tổng không nhỏ hơn `k` phần tử nhỏ nhất; nhưng đề nói "tuyệt đối" nên viết `abs` cũng vô hại.

In [3]:
print(abs(-5), abs(5), abs(0))
print(sum([5, 4]) - sum([2, 2]), abs(sum([2, 2]) - sum([5, 4])))

5 5 0
5 5


### 3.3. `heapq.nlargest` và `heapq.nsmallest`

`heapq.nlargest(k, nums)` trả về `k` phần tử lớn nhất (giảm dần); `heapq.nsmallest(k, nums)` trả về `k` phần tử nhỏ nhất (tăng dần). Phần tử trùng giá trị vẫn xuất hiện nhiều lần. Độ phức tạp `O(n log k)`.

In [4]:
import heapq
nums = [5, 2, 2, 4]
print(heapq.nlargest(2, nums), heapq.nsmallest(2, nums))
print(heapq.nsmallest(3, nums))
print(sum(heapq.nlargest(2, nums)) - sum(heapq.nsmallest(2, nums)))

[5, 4] [2, 2]
[2, 2, 4]
5


### 3.4. `set` làm mất giá trị trùng (và vì sao ở đây là bẫy)

`set(nums)` chỉ giữ giá trị **khác nhau**. Với `[5, 2, 2, 4]`, `set` chỉ còn `{2, 4, 5}`. Đề yêu cầu `k` **phần tử** (theo vị trí), nên hai số `2` là hai phần tử khác nhau.

In [5]:
nums = [5, 2, 2, 4]
print(set(nums), len(set(nums)), len(nums))
print(sorted(set(nums))[:2], "<- thiếu một số 2 so với sorted(nums)[:2] =", sorted(nums)[:2])

{2, 4, 5} 3 4
[2, 4] <- thiếu một số 2 so với sorted(nums)[:2] = [2, 2]


### 3.5. Mảng đếm (counting) cho giá trị nhỏ

Vì `1 <= nums[i] <= 100`, đếm số lần mỗi giá trị bằng `cnt = [0] * 101`. Sau đó "lấy `k` nhỏ nhất" bằng cách duyệt giá trị từ `1` lên, lấy `min(cnt[v], còn_lại)` mỗi lần; "lấy `k` lớn nhất" duyệt từ `100` xuống.

In [6]:
cnt = [0] * 101
for v in [5, 2, 2, 4]:
    cnt[v] += 1
print([(v, c) for v, c in enumerate(cnt) if c])
need, total = 2, 0
for v in range(1, 101):                 # lấy 2 nhỏ nhất
    take = min(cnt[v], need)
    total += v * take
    need -= take
    if need == 0:
        break
print("tổng 2 nhỏ nhất:", total)

[(2, 2), (4, 1), (5, 1)]
tổng 2 nhỏ nhất: 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xếp các phần tử thành hàng từ nhỏ đến lớn. `k` phần tử **ở đầu hàng** là `k` nhỏ nhất; `k` phần tử **ở cuối hàng** là `k` lớn nhất. Hiệu hai tổng chính là "phần chênh lệch" giữa hai đầu hàng.

Tính tay `[5, 2, 2, 4]`, `k = 2`:

- Sắp xếp: `2, 2, 4, 5`
- 2 nhỏ nhất: `2 + 2 = 4`; 2 lớn nhất: `4 + 5 = 9`
- Hiệu: `9 - 4 = 5`

Một nhận xét thú vị: nếu `2k > n` thì hai nhóm **chồng lên nhau** (một số phần tử vừa nằm trong nhóm lớn nhất vừa nằm trong nhóm nhỏ nhất). Khi đó các phần tử chung bị **triệt tiêu** trong hiệu (cộng ở một bên, trừ ở bên kia), và công thức vẫn đúng.

### Bảng tính tay

| nums, k | sắp xếp | k nhỏ nhất | k lớn nhất | kết quả |
|---|---|---|---|---|
| `[5,2,2,4], 2` | 2 2 4 5 | 2, 2 (=4) | 4, 5 (=9) | 5 |
| `[100], 1` | 100 | 100 | 100 | 0 |
| `[1,2,3], 3` | 1 2 3 | 1,2,3 (=6) | 1,2,3 (=6) | 0 |
| `[1,2,3,4,5], 3` | 1 2 3 4 5 | 1,2,3 (=6) | 3,4,5 (=12) | 6 |

In [7]:
for nums, k in [([5, 2, 2, 4], 2), ([100], 1), ([1, 2, 3], 3), ([1, 2, 3, 4, 5], 3)]:
    s = sorted(nums)
    print(nums, k, s, sum(s[:k]), sum(s[-k:]), abs(sum(s[-k:]) - sum(s[:k])))

[5, 2, 2, 4] 2 [2, 2, 4, 5] 4 9 5
[100] 1 [100] 100 100 0
[1, 2, 3] 3 [1, 2, 3] 6 6 0
[1, 2, 3, 4, 5] 3 [1, 2, 3, 4, 5] 6 12 6


## 5. Cách 1 — Brute Force / baseline

Lặp `k` lần: mỗi lần tìm phần tử lớn nhất trong bản sao, cộng vào `big`, rồi **loại một lần xuất hiện** của nó; tương tự với nhỏ nhất. Không cần sắp xếp, chỉ cần `max`, `min`, `remove`.

Nút thắt: mỗi bước `max`, `min`, `remove` đều quét `O(n)`, nên tổng `O(k * n)`. Với `k` lớn bằng `n` thì thành `O(n²)`.

In [8]:
class SolutionBrute:
    def absDifference(self, nums, k):
        a, b = list(nums), list(nums)
        big = small = 0
        for _ in range(k):
            m = max(a); a.remove(m); big += m        # remove chỉ bỏ MỘT lần xuất hiện
            m = min(b); b.remove(m); small += m
        return abs(big - small)

print(SolutionBrute().absDifference([5, 2, 2, 4], 2), SolutionBrute().absDifference([100], 1))

5 0


**Độ phức tạp:** Time `O(k * n)`, Space `O(n)` (hai bản sao).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** sắp xếp một lần, sau đó `k` nhỏ nhất và `k` lớn nhất là hai **đoạn đầu/cuối** của mảng đã sắp xếp. Chỉ cần `sum` hai đoạn đó.

**Quy tắc "nếu… thì…":**

- Nếu `s = sorted(nums)` thì đáp án `= sum(s[-k:]) - sum(s[:k])`.
- Nếu `2k >= n` thì hai đoạn chồng nhau; các phần tử chung triệt tiêu, vẫn đúng công thức.
- Nếu `k = n` thì hai tổng bằng nhau, đáp án `0`.
- Nếu `k = 1` thì đáp án `max - min`.
- Giá trị trùng vẫn tính riêng, nên **không** dùng `set`.

**Invariant:** sau khi sắp xếp, `s[:k]` là `k` phần tử nhỏ nhất và `s[-k:]` là `k` phần tử lớn nhất (theo vị trí).

In [9]:
# Kiểm chứng ý tưởng: sắp xếp+slicing = brute force; đáp án luôn >= 0 (kể cả khi 2k > n)
import random
random.seed(3774)
for _ in range(1500):
    n = random.randint(1, 30)
    nums = [random.randint(1, 100) for _ in range(n)]
    k = random.randint(1, n)
    s = sorted(nums)
    d = sum(s[-k:]) - sum(s[:k])
    assert d >= 0
    assert d == abs(sum(s[-k:]) - sum(s[:k]))
    if k == 1:
        assert d == max(nums) - min(nums)
    if k == n:
        assert d == 0
print("sum(s[-k:]) - sum(s[:k]) luôn >= 0; k=1 -> max-min; k=n -> 0")

sum(s[-k:]) - sum(s[:k]) luôn >= 0; k=1 -> max-min; k=n -> 0


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + slicing

`s = sorted(nums)` rồi `sum(s[-k:]) - sum(s[:k])`. Rõ ràng và là lời giải chuẩn.

In [10]:
class SolutionSort:
    def absDifference(self, nums, k):
        s = sorted(nums)
        return abs(sum(s[-k:]) - sum(s[:k]))

print(SolutionSort().absDifference([5, 2, 2, 4], 2), SolutionSort().absDifference([100], 1))

5 0


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`.

### Cách 3 — `heapq.nlargest` / `heapq.nsmallest`

Không cần sắp xếp toàn bộ: dùng heap để lấy đúng `k` phần tử mỗi phía. Tốt khi `k` nhỏ so với `n` (`O(n log k)`).

In [11]:
import heapq

class SolutionHeap:
    def absDifference(self, nums, k):
        return abs(sum(heapq.nlargest(k, nums)) - sum(heapq.nsmallest(k, nums)))

print(SolutionHeap().absDifference([5, 2, 2, 4], 2), SolutionHeap().absDifference([100], 1))

5 0


**Độ phức tạp:** Time `O(n log k)`, Space `O(k)`.

### Cách 4 — Mảng đếm (counting)

Vì giá trị `<= 100`, ta đếm một lần rồi lấy `k` phần tử từ đầu (nhỏ) và từ cuối (lớn) của bảng đếm. `O(n + 100)`, không sắp xếp.

In [12]:
class SolutionCount:
    def absDifference(self, nums, k):
        cnt = [0] * 101
        for v in nums:
            cnt[v] += 1
        small = 0
        need = k
        for v in range(1, 101):                   # k nhỏ nhất: từ giá trị nhỏ lên
            take = min(cnt[v], need)
            small += v * take
            need -= take
            if need == 0:
                break
        big = 0
        need = k
        for v in range(100, 0, -1):               # k lớn nhất: từ giá trị lớn xuống
            take = min(cnt[v], need)
            big += v * take
            need -= take
            if need == 0:
                break
        return abs(big - small)

print(SolutionCount().absDifference([5, 2, 2, 4], 2), SolutionCount().absDifference([100], 1))

5 0


**Độ phức tạp:** Time `O(n + 100)`, Space `O(1)` (bảng 101 ô).

### Cách 5 — Hai con trỏ trên mảng đã sắp xếp

Sắp xếp, rồi cho `i` đi từ đầu và `j` đi từ cuối đúng `k` bước, cộng dồn `s[j] - s[i]`. Mỗi bước cộng hiệu của một cặp "lớn - nhỏ" tương ứng; tổng các hiệu đúng bằng hiệu hai tổng.

In [13]:
class SolutionTwoPointers:
    def absDifference(self, nums, k):
        s = sorted(nums)
        i, j = 0, len(s) - 1
        diff = 0
        for _ in range(k):
            diff += s[j] - s[i]                   # cặp (lớn thứ t) - (nhỏ thứ t)
            i += 1
            j -= 1
        return abs(diff)

print(SolutionTwoPointers().absDifference([5, 2, 2, 4], 2), SolutionTwoPointers().absDifference([100], 1))

5 0


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`. Chú ý: khi `2k > n`, `i` vượt `j` và các cặp "đảo ngược" cho hiệu âm; tổng các hiệu vẫn bằng hiệu hai tổng (các số hạng chung triệt tiêu theo từng cặp), nên cách này vẫn đúng.

## 8. TRACE từng bước

Trace Cách 5 (hai con trỏ) với `nums = [5, 2, 2, 4, 1]`, `k = 2`: `s = [1, 2, 2, 4, 5]`.

| bước | `i` | `j` | `s[i]` | `s[j]` | cộng `s[j] - s[i]` | `diff` |
|---|---|---|---|---|---|---|
| 1 | 0 | 4 | 1 | 5 | 4 | 4 |
| 2 | 1 | 3 | 2 | 4 | 2 | 6 |

Kiểm tra bằng hiệu hai tổng: `(4 + 5) - (1 + 2) = 6`.

In [14]:
def trace_two_pointers(nums, k):
    s = sorted(nums)
    print("sắp xếp:", s, " k =", k)
    i, j, diff = 0, len(s) - 1, 0
    for step in range(1, k + 1):
        add = s[j] - s[i]
        diff += add
        print(f"  bước {step}: i={i} j={j}  s[i]={s[i]} s[j]={s[j]}  cộng {add}  diff={diff}")
        i += 1
        j -= 1
    print("  kiểm tra:", sum(s[-k:]), "-", sum(s[:k]), "=", sum(s[-k:]) - sum(s[:k]))
    return abs(diff)

assert trace_two_pointers([5, 2, 2, 4, 1], 2) == 6
assert trace_two_pointers([5, 2, 2, 4], 2) == 5

sắp xếp: [1, 2, 2, 4, 5]  k = 2
  bước 1: i=0 j=4  s[i]=1 s[j]=5  cộng 4  diff=4
  bước 2: i=1 j=3  s[i]=2 s[j]=4  cộng 2  diff=6
  kiểm tra: 9 - 3 = 6
sắp xếp: [2, 2, 4, 5]  k = 2
  bước 1: i=0 j=3  s[i]=2 s[j]=5  cộng 3  diff=3
  bước 2: i=1 j=2  s[i]=2 s[j]=4  cộng 2  diff=5
  kiểm tra: 9 - 4 = 5


In [15]:
def trace_count(nums, k):
    cnt = [0] * 101
    for v in nums:
        cnt[v] += 1
    print("bảng đếm khác 0:", {v: c for v, c in enumerate(cnt) if c})
    need, small = k, 0
    for v in range(1, 101):
        if need == 0:
            break
        take = min(cnt[v], need)
        if take:
            print(f"  nhỏ: giá trị {v} lấy {take} -> còn cần {need - take}")
        small += v * take
        need -= take
    need, big = k, 0
    for v in range(100, 0, -1):
        if need == 0:
            break
        take = min(cnt[v], need)
        if take:
            print(f"  lớn: giá trị {v} lấy {take} -> còn cần {need - take}")
        big += v * take
        need -= take
    print(f"  tổng nhỏ={small}, tổng lớn={big}")
    return abs(big - small)

assert trace_count([5, 2, 2, 4], 2) == 5

bảng đếm khác 0: {2: 2, 4: 1, 5: 1}
  nhỏ: giá trị 2 lấy 2 -> còn cần 0
  lớn: giá trị 5 lấy 1 -> còn cần 1
  lớn: giá trị 4 lấy 1 -> còn cần 0
  tổng nhỏ=4, tổng lớn=9


## 9. Code nộp LeetCode

In [16]:
from typing import List, Optional

class Solution:
    def absDifference(self, nums: List[int], k: int) -> int:
        s = sorted(nums)
        return abs(sum(s[-k:]) - sum(s[:k]))

## 10. Vì sao thuật toán đúng

- **Đoạn đầu/cuối là nhóm cực trị:** trong mảng đã sắp xếp tăng dần, bất kỳ `k` phần tử nào cũng có tổng `>= sum(s[:k])` (vì phần tử thứ `t` được chọn không nhỏ hơn `s[t-1]`) và `<= sum(s[-k:])`. Do đó `k` nhỏ nhất có tổng `sum(s[:k])` và `k` lớn nhất có tổng `sum(s[-k:])`.
- **Giá trị trùng:** sắp xếp giữ đủ `n` phần tử (kể cả trùng), nên đoạn cắt lấy đúng `k` **vị trí**, khớp với cách đề đếm phần tử.
- **Không âm:** `sum(s[-k:]) >= sum(s[:k])` nên `abs` không đổi giá trị; khi `k = n` hai tổng bằng nhau, hiệu `0`.
- **Chồng lấn (`2k > n`):** các phần tử thuộc cả hai nhóm có mặt trong cả hai tổng nên triệt tiêu khi trừ; công thức vẫn là hiệu của hai tổng đã định nghĩa, nên đúng theo đề.
- **Các cách còn lại** cho cùng hai tổng (heap, mảng đếm) hoặc cùng hiệu (hai con trỏ), nên cùng đáp án.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Loại dần max/min | `O(k * n)` | `O(n)` |
| Sắp xếp + slicing | `O(n log n)` | `O(n)` |
| `heapq` | `O(n log k)` | `O(k)` |
| Mảng đếm | `O(n + V)` với `V = 100` | `O(V)` |
| Hai con trỏ | `O(n log n)` | `O(n)` |

`n = len(nums)`, `V = 100` là giá trị lớn nhất. Với `n <= 100` mọi cách đều tức thì. Đáp án tối đa không quá `k * 99 <= 9900`, nằm gọn trong `int`.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, `k = 1`, `k = n`, `2k > n`, mọi phần tử bằng nhau, `n = 100`, giá trị `1` và `100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [17]:
impls = [SolutionBrute(), SolutionSort(), SolutionHeap(), SolutionCount(), SolutionTwoPointers(), Solution()]

cases = [
    (([5, 2, 2, 4], 2), 5), (([100], 1), 0), (([1, 2, 3], 3), 0), (([1, 2, 3, 4, 5], 3), 6),
    (([7, 7, 7, 7], 2), 0), (([1, 100], 1), 99), (([100, 1], 1), 99), (([1, 100], 2), 0),
    (([1, 1, 100, 100], 2), 198), (([1, 2, 3, 4], 3), 3), ((list(range(1, 101)), 100), 0),
    (([1] * 50 + [100] * 50, 50), 99 * 50), (([1] * 50 + [100] * 50, 1), 99),
]
for (nums, k), exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 100 for v in nums) and 1 <= k <= len(nums)
    for impl in impls:
        got = impl.absDifference(list(nums), k)
        assert got == exp, (type(impl).__name__, nums, k, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 13 ca


In [18]:
import random
random.seed(37740)
for _ in range(1000):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    k = random.randint(1, n)
    s = sorted(nums)
    exp = abs(sum(s[-k:]) - sum(s[:k]))
    outs = [impl.absDifference(list(nums), k) for impl in impls]
    assert all(o == exp for o in outs), (nums, k, outs, exp)
    assert exp >= 0 and exp <= (max(nums) - min(nums)) * k
print("1000 test ngẫu nhiên (seed=37740): 6 cách giống nhau")

1000 test ngẫu nhiên (seed=37740): 6 cách giống nhau


In [19]:
# Tính chất: bất biến theo thứ tự; k tăng thì đáp án không giảm; k=1 cho max-min
for _ in range(300):
    n = random.randint(1, 40)
    nums = [random.randint(1, 100) for _ in range(n)]
    vals = [Solution().absDifference(nums, k) for k in range(1, n + 1)]
    assert vals[0] == max(nums) - min(nums) and vals[-1] == 0
    random.shuffle(nums)
    assert Solution().absDifference(nums, 1) == vals[0]
    h = n // 2
    assert all(vals[i] <= vals[i + 1] for i in range(h - 1)) if h >= 2 else True
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `set` để khử trùng

Đề đếm phần tử theo **vị trí**; `set` làm mất giá trị trùng và đổi đáp án.

In [20]:
def wrong_set(nums, k):
    s = sorted(set(nums))                  # SAI: mất bản sao của giá trị trùng
    return abs(sum(s[-k:]) - sum(s[:k]))

print(wrong_set([5, 2, 2, 4], 2), "!=", Solution().absDifference([5, 2, 2, 4], 2))
assert wrong_set([5, 2, 2, 4], 2) == 3 != 5

3 != 5


### Lỗi 2: quên sắp xếp trước khi cắt

`nums[:k]` và `nums[-k:]` chỉ là `k` phần tử đầu/cuối **theo thứ tự ban đầu**, không phải nhỏ/lớn nhất.

In [21]:
def wrong_no_sort(nums, k):
    return abs(sum(nums[-k:]) - sum(nums[:k]))    # SAI: nums chưa sắp xếp

print(wrong_no_sort([5, 2, 2, 4], 2), "!=", Solution().absDifference([5, 2, 2, 4], 2))
assert wrong_no_sort([5, 2, 2, 4], 2) == 1 != 5

1 != 5


### Lỗi 3: so sánh phần tử thứ `k` thay vì **tổng** `k` phần tử

Đề cộng **cả `k` phần tử** mỗi phía, không chỉ lấy phần tử thứ `k`.

In [22]:
def wrong_kth(nums, k):
    s = sorted(nums)
    return abs(s[-k] - s[k - 1])                   # SAI: chỉ phần tử thứ k

print(wrong_kth([5, 2, 2, 4], 2), "!=", Solution().absDifference([5, 2, 2, 4], 2))
assert wrong_kth([5, 2, 2, 4], 2) == 2 != 5

2 != 5


### Lỗi 4: dùng `k = 0` kiểu slicing `s[-0:]`

Nếu ai đó viết hàm tổng quát cho `k = 0` mà dùng `s[-k:]`, kết quả lấy cả list. Đề cho `k >= 1` nên không xảy ra ở bài này, nhưng đây là bẫy kinh điển khi tái sử dụng.

In [23]:
def risky_last_k(s, k):
    return s[-k:]                                  # SAI khi k = 0

print(risky_last_k([1, 2, 3], 0), "<- lấy cả list thay vì rỗng")
def safe_last_k(s, k):
    return s[len(s) - k:] if k else []
assert risky_last_k([1, 2, 3], 0) == [1, 2, 3] != safe_last_k([1, 2, 3], 0) == []
assert safe_last_k([1, 2, 3], 2) == [2, 3]

[1, 2, 3] <- lấy cả list thay vì rỗng


### Lỗi 5: sửa mảng đầu vào bằng `nums.sort()` rồi dùng lại

`list.sort()` sửa tại chỗ. Nếu bạn cần mảng gốc sau đó (hoặc chạy hai cách trên cùng một list), kết quả có thể sai hoặc khó tìm lỗi. `sorted(nums)` an toàn hơn.

In [24]:
def sorts_in_place(nums, k):
    nums.sort()                                    # có tác dụng phụ lên đầu vào
    return abs(sum(nums[-k:]) - sum(nums[:k]))

data = [5, 2, 2, 4]
sorts_in_place(data, 2)
print("đầu vào sau khi gọi:", data)
data2 = [5, 2, 2, 4]
Solution().absDifference(data2, 2)
assert data != [5, 2, 2, 4] and data2 == [5, 2, 2, 4]

đầu vào sau khi gọi: [2, 2, 4, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng `k` lớn nhất / nhỏ nhất" -> sắp xếp rồi slicing; hoặc `heapq.nlargest/nsmallest` khi `k` nhỏ.
- Giá trị trùng nhau: **đừng** dùng `set` nếu đề tính theo phần tử/vị trí.
- Đáp án luôn `>= 0`; `k = 1` cho `max - min`; `k = n` cho `0`.
- Giá trị nhỏ và bị chặn (<= 100) cho phép mảng đếm `O(n + V)`.
- Một dòng nộp nhanh: `s = sorted(nums); return abs(sum(s[-k:]) - sum(s[:k]))`.
- Kiểm tra bằng ví dụ 1: nếu bạn ra `3` thay vì `5` là đã khử trùng nhầm.

## 15. Biến thể của bài

1. **Kth Largest Element in an Array** (LeetCode 215): phần tử lớn thứ `k` (quickselect hoặc heap).
2. **Top K Frequent Elements** (347): `k` giá trị xuất hiện nhiều nhất.
3. **Maximum Product of Two Elements** (1464): chọn hai phần tử cho tích lớn nhất.
4. **Minimum Difference Between Highest and Lowest of K Scores** (1984): chọn `k` điểm sao cho hiệu max-min nhỏ nhất (sắp xếp + cửa sổ).
5. **Sum of k Smallest Pairs / Find K Pairs with Smallest Sums** (373): tổng cặp nhỏ nhất bằng heap.
6. **Maximize Sum of Array After K Negations** (1005): `k` lần đổi dấu để tổng lớn nhất (tham lam trên mảng đã sắp xếp).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Loại dần max/min | `k` lần `max`/`min` + `remove` | `O(k * n)` | `O(n)` | Oracle kiểm tra, `k` nhỏ |
| Sắp xếp + slicing | `sum(s[-k:]) - sum(s[:k])` | `O(n log n)` | `O(n)` | Nộp bài, rõ ràng |
| `heapq` | `nlargest` / `nsmallest` | `O(n log k)` | `O(k)` | `k` nhỏ so với `n` |
| Mảng đếm | đếm giá trị `<= 100` | `O(n + V)` | `O(V)` | Giá trị bị chặn nhỏ |
| Hai con trỏ | cộng `s[j] - s[i]` k bước | `O(n log n)` | `O(n)` | Minh họa cặp lớn-nhỏ |

Nộp cách sắp xếp + slicing.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().absDifference([3, 1, 4, 1, 5], 2)` ra gì?

**Bài 2 (tính tay):** với `nums = [10, 20, 30, 40, 50], k = 3`, hai nhóm chồng lên nhau ở đâu và đáp án là bao nhiêu?

**Bài 3 (điền TODO):** hoàn thành `sum_of_k_largest(nums, k)` bằng `heapq`.

**Bài 4 (tự viết lại):** viết `min_diff_k_scores(nums, k)` (LeetCode 1984): chọn `k` phần tử sao cho `max - min` của chúng nhỏ nhất.

**Bài 5 (mini challenge):** viết `abs_difference_distinct(nums, k)` phiên bản **chỉ xét giá trị khác nhau** (dùng `set`), và chỉ ra ví dụ làm nó khác `absDifference`.

In [25]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().absDifference([3, 1, 4, 1, 5], 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def sum_of_k_largest_todo(nums, k):
    import heapq
    # TODO: dùng heapq.nlargest(k, nums)
    return 0

7


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: sắp xếp thành `1, 1, 3, 4, 5`. Bài 2: `n = 5`, `k = 3` nên `2k = 6 > 5`.

**Mức 2 (vừa):** Bài 3: `sum(heapq.nlargest(k, nums))`. Bài 4: sau khi sắp xếp, đáp án là `min(s[i + k - 1] - s[i])` với mọi `i`. Bài 5: dùng `sorted(set(nums))` rồi cắt `k` phần tử mỗi đầu.

**Mức 3 (gần đáp án):** Bài 1: nhỏ nhất `1 + 1 = 2`, lớn nhất `4 + 5 = 9`, đáp án `7`. Bài 2: nhóm lớn `{30,40,50}` (120), nhóm nhỏ `{10,20,30}` (60), chung phần tử `30`, đáp án `60`. Bài 5: `[5,2,2,4], k=2` cho `3` thay vì `5`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
import heapq

# Bài 1, 2
assert Solution().absDifference([3, 1, 4, 1, 5], 2) == 7
assert Solution().absDifference([10, 20, 30, 40, 50], 3) == 120 - 60 == 60

# Bài 3
def sum_of_k_largest(nums, k):
    return sum(heapq.nlargest(k, nums))
assert sum_of_k_largest([5, 2, 2, 4], 2) == 9 and sum_of_k_largest([100], 1) == 100

# Bài 4
def min_diff_k_scores(nums, k):
    s = sorted(nums)
    return min(s[i + k - 1] - s[i] for i in range(len(s) - k + 1))
assert min_diff_k_scores([90], 1) == 0 and min_diff_k_scores([9, 4, 1, 7], 2) == 2
assert min_diff_k_scores([1, 5, 9, 13], 3) == 8

# Bài 5
def abs_difference_distinct(nums, k):
    s = sorted(set(nums))
    return abs(sum(s[-k:]) - sum(s[:k]))
assert abs_difference_distinct([5, 2, 2, 4], 2) == 3 != Solution().absDifference([5, 2, 2, 4], 2) == 5
assert abs_difference_distinct([1, 2, 3, 4, 5], 2) == Solution().absDifference([1, 2, 3, 4, 5], 2)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng k lớn nhất / k nhỏ nhất":**

```python
s = sorted(nums)
return abs(sum(s[-k:]) - sum(s[:k]))
# k nhỏ so với n: heapq.nlargest(k, nums) / heapq.nsmallest(k, nums)
```

**Checklist trước khi submit:**

- [ ] Đã **sắp xếp** trước khi cắt đầu/cuối?
- [ ] Giá trị trùng được tính theo từng vị trí (không dùng `set`)?
- [ ] Tính **tổng** `k` phần tử mỗi phía, không chỉ phần tử thứ `k`?
- [ ] Kết quả không âm (`abs` hoặc đúng thứ tự trừ)?
- [ ] Thử `k = 1`, `k = n`, `n = 1` và `2k > n`?
- [ ] Chữ ký đúng `absDifference(self, nums: List[int], k: int) -> int`?